## Charged tensor operations

A symmetric MPS is a network of intertwiners: every tensor is invariant, and the charge of the
state is carried by the bonds.  An operator that carries a charge of its own -- the spin-1 part of
a bond operator, built here as an MPO with `total=(2,)` -- moves the state into another sector, so
the bonds of the result have to carry the charge it adds.

The engine keeps two representations of the same tensors and this notebook applies the operator in
both:

* the block-sparse (reduced) representation, whose blocks are labelled by their fusion path, and
* the resolved representation, `state.resolve()`, whose blocks store the dense window of their legs
  with all fusion paths summed.

Both must reproduce the plain dense application, and the projection of the resolved data must
recover the reduced labelling exactly.

In [ ]:
import torch

from base import COMPLEX, CUDA
from mpsdmrg import Broomstick, MPO, MPS
from mpsdmrg.symmetry.group import su2_symmetry, su3_symmetry
from mpsdmrg.tensor import Tensor, resolve

LENGTH, PHYS = 6, 2


def bond_table(state: MPS) -> None:
    '''
    Print the sectors, their multiplicities, and the number of states on every bond.

    Args:
        state: the state whose bond legs are read, the left edge included.
    '''
    print(f'{"bond":>5}{"multiplets":>12}{"states":>8}   blocks')
    for index, tensor in enumerate(state.tensors):
        leg = tensor.legs[2]
        blocks = '  '.join(f'{leg.sector(block)} x{leg.multiplicity(block)}'
                           for block in range(leg.nblocks))
        print(f'{index + 1:>5}{leg.multiplets:>12}{leg.dim:>8}   {blocks}')


stick = Broomstick(L=LENGTH, phys_dim=PHYS, max_bond_dim=32, svd_tol=1e-14, group=su2_symmetry)
for site in range(LENGTH - 1):
    for axis in 'XYZ':
        stick.add_couplings((axis, axis), 1.0, (site, site + 1))
stick.build()
stick.sweep(num_sweeps=6)
state = stick.state

print(f'Heisenberg L={LENGTH}: energy {stick.energy:.12f}, per bond {stick.energy / LENGTH:.12f}')
bond_table(state)
print('bond_dim (multiplets)', state.bond_dim)

The bond operator $\vec S_i \cdot \vec S_{i+1}$ of the Heisenberg chain is invariant, so every MPO
tensor of the Hamiltonian is neutral on all of its legs.  Placing the same three couplings in the
charge channel `total=(2,)` builds an operator that carries the triplet charge instead.  The dense
reference contracts the MPO into a $2^L \times 2^L$ matrix and applies it to the dense state
vector; the engine applies the MPO to the MPS in either representation, and the bonds of the result
carry the charge the operator added.

In [ ]:
ALPHA = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'


def dense_state(state: MPS) -> torch.Tensor:
    '''
    Contract the MPS tensors into one vector.

    Args:
        state: the state to contract.
    Returns:
        The dense vector in shape (phys ** L), the site 0 physical index most significant.
    '''
    length = state.L
    bond, phys = ALPHA[:length + 1], ALPHA[length + 1:2 * length + 1]
    expression = ','.join(f'{bond[i]}{phys[i]}{bond[i + 1]}' for i in range(length))
    return torch.einsum(expression + '->' + ''.join(phys),
                        *[tensor.to_torch() for tensor in state.tensors]).reshape(-1)


def dense_operator(operator: MPO, dim: int) -> torch.Tensor:
    '''
    Contract the MPO tensors into one matrix over the physical leg.

    Args:
        operator: the MPO to contract; its tensors carry the legs (left, right, phys, phys.conj()).
        dim: the physical dimension of one site.
    Returns:
        The dense matrix in shape (dim ** L, dim ** L), row bra and column ket.
    '''
    length = operator.L
    ket, bra = ALPHA[:length], ALPHA[length:2 * length]
    bond = ALPHA[2 * length:3 * length + 1]
    expression = ','.join(f'{bond[i]}{bond[i + 1]}{ket[i]}{bra[i]}' for i in range(length))
    matrix = torch.einsum(expression + '->' + ''.join(bra) + ''.join(ket),
                          *[tensor.to_torch() for tensor in operator.tensors])
    return matrix.reshape(dim ** length, dim ** length)


def charge_mpo(resolved: bool) -> MPO:
    '''
    The triplet part of the bond operator on sites (0, 1).

    Args:
        resolved: move the MPO tensors into the resolved representation.
    Returns:
        The MPO carrying the three Heisenberg couplings in the charge channel total=(2,).
    '''
    operator = MPO(L=LENGTH, phys_dim=PHYS, group=su2_symmetry, total=(2,))
    for axis in 'XYZ':
        operator.add_couplings((axis, axis), 1.0, (0, 1))
    operator.build()
    if resolved:
        operator.tensors = [resolve(tensor) for tensor in operator.tensors]
    return operator


charge = charge_mpo(resolved=False)
reference = float((dense_operator(charge, PHYS) @ dense_state(state)).abs().pow(2).sum())
applied = state.apply(charge, max_bond_dim=32, tol=1e-14)
resolved_applied = state.resolve().apply(charge_mpo(resolved=True), max_bond_dim=32, tol=1e-14)

print(f'{"":<14}{"|O psi|^2":>22}{"|diff|":>12}')
print(f'{"dense":<14}{reference:>22.15f}')
print(f'{"block-sparse":<14}{float(applied.norm().real):>22.15f}'
      f'{abs(float(applied.norm().real) - reference):>12.2e}')
print(f'{"resolved":<14}{float(resolved_applied.norm().real):>22.15f}'
      f'{abs(float(resolved_applied.norm().real) - reference):>12.2e}')
bond_table(applied)
print('bond_dim (multiplets)', applied.bond_dim)

The resolved representation is what a bare matrix needs: `Tensor.unresolve()` projects a resolved
block back onto the reduced labelling, one amplitude per fusion path and per multiplet copy, and
that projection is what makes a split of a resolved tensor exact.  The round trip below is expected
to be exact to machine precision, and a bare $S^z$ applied to a resolved state must reproduce the
engine's expectation $\langle (S^z)^2 \rangle$, staying in the resolved representation.

In [ ]:
tensor = state.tensors[2]
round_trip = resolve(tensor).unresolve().to_torch() - tensor.to_torch()
print('reduced -> resolved -> reduced   max|diff|', float(round_trip.abs().max()))

s_z = torch.tensor([[0.5, 0.0], [0.0, -0.5]], dtype=COMPLEX, device=CUDA)
magnetization = complex(state.expectation({0: s_z @ s_z})).real
rotated = state.resolve().apply(s_z, [0], max_bond_dim=32, tol=1e-14)
print(f'{"":<22}{"norm squared":>22}')
print(f'{"<S_z^2>":<22}{magnetization:>22.15f}')
print(f'{"resolved S_z state":<22}{float(rotated.norm().real):>22.15f}')
print(f'{"difference":<22}{abs(float(rotated.norm().real) - magnetization):>22.3e}')
print('every leg resolved',
      all(leg.resolved for tensor in rotated.tensors for leg in tensor.legs))

SU(3) is not multiplicity-free: the adjoint appears twice in $8 \otimes 8$, so a bond can carry the
same irrep in more than one multiplet.  The ULS chain therefore exercises the projection over
repeated fusion channels as well as over the multiplet copies of a charge; the table below shows
the multiplicity of every block on every bond of the result.

In [ ]:
def gell_mann() -> list[torch.Tensor]:
    '''
    The eight Hermitian Gell-Mann generators of SU(3).

    Returns:
        The generators in shape (3, 3), normalized so that tr(T_a T_b) = 2 delta_ab.
    '''
    imaginary = 1j
    raw = [
        [[0, 1, 0], [1, 0, 0], [0, 0, 0]],
        [[0, -imaginary, 0], [imaginary, 0, 0], [0, 0, 0]],
        [[1, 0, 0], [0, -1, 0], [0, 0, 0]],
        [[0, 0, 1], [0, 0, 0], [1, 0, 0]],
        [[0, 0, -imaginary], [0, 0, 0], [imaginary, 0, 0]],
        [[0, 0, 0], [0, 0, 1], [0, 1, 0]],
        [[0, 0, 0], [0, 0, -imaginary], [0, imaginary, 0]],
        [[1, 0, 0], [0, 1, 0], [0, 0, -2]],
    ]
    return [torch.tensor(matrix, dtype=COMPLEX, device=CUDA) / (1.0 if index < 7 else 3 ** 0.5)
            for index, matrix in enumerate(raw)]


DIM3 = 3
mapping = {f'L{index + 1}': Tensor.from_torch(generator)
           for index, generator in enumerate(gell_mann())}
mapping['I'] = Tensor.from_torch(torch.eye(DIM3, dtype=COMPLEX, device=CUDA))

uls = Broomstick(L=LENGTH, phys_dim=DIM3, max_bond_dim=32, svd_tol=1e-14, mapping=mapping,
                 group=su3_symmetry)
for site in range(LENGTH - 1):
    uls.add_couplings(('I', 'I'), 1.0 / 3.0, (site, site + 1))
    for name in mapping:
        if name != 'I':
            uls.add_couplings((name, name), 0.5, (site, site + 1))
uls.build()
uls.sweep(num_sweeps=4)

adjoint = MPO(L=LENGTH, phys_dim=DIM3, group=su3_symmetry, total=(1, 1), mapping=mapping)
adjoint.add_couplings(('L1', 'L1'), 1.0, (0, 1))
adjoint.build()

reference3 = float((dense_operator(adjoint, DIM3) @ dense_state(uls.state)).abs().pow(2).sum())
applied3 = uls.state.apply(adjoint, max_bond_dim=32, tol=1e-14)
print(f'ULS L={LENGTH}: energy {uls.energy:.12f}, per bond {uls.energy / LENGTH:.12f}')
print(f'{"":<14}{"|O psi|^2":>22}{"|diff|":>12}')
print(f'{"dense":<14}{reference3:>22.15f}')
print(f'{"block-sparse":<14}{float(applied3.norm().real):>22.15f}'
      f'{abs(float(applied3.norm().real) - reference3):>12.2e}')
bond_table(applied3)